# Lesson 3.4 — Values Schema Validation

**Phase 3: Advanced Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Create a `values.schema.json` to validate chart inputs
- Catch misconfiguration at install/upgrade time before anything is deployed
- Use JSON Schema to enforce types, required fields, and enum values

---

## Why Schema Validation?

Without a schema, Helm accepts any values — including typos or wrong types. A schema lets you:

- Reject `replicaCount: "two"` (should be an integer)
- Require `image.repository` to always be set
- Constrain `service.type` to only valid Kubernetes types
- Provide a self-documenting contract for chart users

Schema validation runs on `helm install`, `helm upgrade`, `helm lint`, and `helm template`.

> **Reference:** [Charts — Schema Files](https://helm.sh/docs/topics/charts/#schema-files) — https://helm.sh/docs/topics/charts/#schema-files

---

## Setup

In [ ]:
rm -rf /tmp/schemademo && mkdir -p /tmp/schemademo/templates

cat > /tmp/schemademo/Chart.yaml << 'EOF'
apiVersion: v2
name: schemademo
version: 0.1.0
appVersion: "1.0"
EOF

cat > /tmp/schemademo/values.yaml << 'EOF'
replicaCount: 1

image:
  repository: nginx
  tag: "1.25"
  pullPolicy: IfNotPresent

service:
  type: ClusterIP
  port: 80

ingress:
  enabled: false
EOF

# A minimal template to render
cat > /tmp/schemademo/templates/configmap.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-config
data:
  replicas: {{ .Values.replicaCount | quote }}
  serviceType: {{ .Values.service.type | quote }}
EOF

echo "Setup complete"

---

## Writing `values.schema.json`

The schema uses [JSON Schema Draft-07](https://json-schema.org/specification-links#draft-7).

In [ ]:
cat > /tmp/schemademo/values.schema.json << 'EOF'
{
  "$schema": "https://json-schema.org/draft-07/schema#",
  "title": "Schemademo Chart Values",
  "type": "object",
  "required": ["replicaCount", "image", "service"],
  "properties": {
    "replicaCount": {
      "description": "Number of pod replicas",
      "type": "integer",
      "minimum": 1,
      "maximum": 20
    },
    "image": {
      "description": "Container image settings",
      "type": "object",
      "required": ["repository"],
      "properties": {
        "repository": {
          "type": "string",
          "minLength": 1
        },
        "tag": {
          "type": "string"
        },
        "pullPolicy": {
          "type": "string",
          "enum": ["Always", "IfNotPresent", "Never"],
          "description": "Must be one of: Always, IfNotPresent, Never"
        }
      }
    },
    "service": {
      "description": "Kubernetes Service settings",
      "type": "object",
      "required": ["type", "port"],
      "properties": {
        "type": {
          "type": "string",
          "enum": ["ClusterIP", "NodePort", "LoadBalancer", "ExternalName"]
        },
        "port": {
          "type": "integer",
          "minimum": 1,
          "maximum": 65535
        }
      }
    },
    "ingress": {
      "type": "object",
      "properties": {
        "enabled": {
          "type": "boolean"
        }
      }
    }
  }
}
EOF

echo "Schema written"

---

## Testing the Schema

### Valid Values — Should Succeed

In [ ]:
# Should succeed — valid values
helm template my-release /tmp/schemademo --set replicaCount=3
echo "Success!"

### Invalid Type — Should Fail

In [ ]:
# replicaCount must be an integer — passing a string should fail
helm template my-release /tmp/schemademo --set replicaCount="two" 2>&1 || true

In [ ]:
# replicaCount must be >= 1 and <= 20 — passing 0 should fail
helm template my-release /tmp/schemademo --set replicaCount=0 2>&1 || true

In [ ]:
# pullPolicy must be one of the enum values
helm template my-release /tmp/schemademo --set image.pullPolicy=sometimes 2>&1 || true

In [ ]:
# service.type must be a valid Kubernetes service type
helm template my-release /tmp/schemademo --set service.type=Ingress 2>&1 || true

---

## Schema + `helm lint`

`helm lint` also runs schema validation, making it great for CI:

In [ ]:
# Lint with valid values — should pass
helm lint /tmp/schemademo

echo ""

# Lint with an invalid values file
cat > /tmp/bad-values.yaml << 'EOF'
replicaCount: "not-a-number"  # INVALID: should be integer
service:
  type: InvalidType             # INVALID: not in enum
  port: 80
image:
  repository: nginx
EOF

helm lint /tmp/schemademo -f /tmp/bad-values.yaml 2>&1 || true

---

## JSON Schema Quick Reference

| Keyword | Purpose | Example |
|---|---|---|
| `type` | Data type | `"string"`, `"integer"`, `"boolean"`, `"object"`, `"array"` |
| `required` | Mandatory keys in an object | `["image", "service"]` |
| `minimum` / `maximum` | Numeric bounds | `"minimum": 1` |
| `minLength` / `maxLength` | String length bounds | `"minLength": 1` |
| `enum` | Allowed values | `["ClusterIP", "NodePort"]` |
| `pattern` | Regex for strings | `"^[a-z]+$"` |
| `description` | Documentation string | Shown in error messages |
| `default` | Default value hint (informational) | `"default": 1` |

---

## Exercise

In [ ]:
# TASK 1: Add a 'resources' section to values.yaml:
#   resources:
#     limits:
#       cpu: "100m"
#       memory: "128Mi"
#
# Then add schema validation for 'resources' that:
#   - Requires cpu to be a string matching pattern '^[0-9]+m$'
#   - Requires memory to be a string matching pattern '^[0-9]+(Mi|Gi)$'
# TODO:


# TASK 2: Add an 'environment' string value constrained to
#         enum: ["development", "staging", "production"]
# TODO:


# TASK 3: Run helm lint with each of the following and capture the output:
#   a) Valid values (all constraints satisfied)
#   b) resources.limits.cpu: 100 (integer instead of string)
#   c) environment: test (not in enum)
# TODO:

echo "Exercise: implement the tasks above"

---

## Summary

- `values.schema.json` at the chart root adds JSON Schema validation to values.
- Validation runs on `helm install`, `helm upgrade`, `helm template`, and `helm lint`.
- Use `type`, `required`, `enum`, `minimum`/`maximum`, and `pattern` to catch mistakes early.
- `helm lint -f values.yaml` is a great CI check to run before every release.

---

## Next Lesson

➡️ [Lesson 4.1 — Chart Dependencies](../phase-4-dependencies-hooks/lesson-4.1-chart-dependencies.ipynb)